# 📊 Chapter 12: Descriptive Statistics
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** A company reports "average salary of $85,000." Half the employees make less than $60,000.
> Both statements are technically true. Descriptive statistics tells you which story to trust.

**🎯 Objectives:** Compute and interpret measures of center, spread, shape, and outlier detection.

## 📖 Section 1 — Concept Review

### Measures of Center
| Measure | Formula | Best When |
|---------|---------|-----------|
| **Mean** | Σxᵢ/n | Symmetric, no outliers |
| **Median** | Middle value | Skewed data, outliers present |
| **Mode** | Most frequent | Categorical, multimodal |

### Measures of Spread
| Measure | Formula | Best When |
|---------|---------|-----------|
| **Range** | max-min | Quick overview |
| **IQR** | Q3-Q1 | Robust, outliers present |
| **Variance** | Σ(xᵢ-μ)²/n | Mathematical analysis |
| **Std Dev** | √Variance | Same units as data |

### Shape: Skewness & Kurtosis
- **Skewness > 0**: Right tail (mean > median) — income, housing prices
- **Skewness < 0**: Left tail (mean < median) — exam scores
- **Kurtosis**: How heavy the tails are (normal = 3)

### Outlier Detection: IQR Method
Outlier if: x < Q1 - 1.5×IQR  OR  x > Q3 + 1.5×IQR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
sns.set_theme(style="whitegrid")
np.random.seed(42)

# Realistic income data (right-skewed)
incomes = np.concatenate([
    np.random.lognormal(10.8, 0.4, 950),   # typical workers
    np.random.lognormal(13, 0.5, 50),       # executives
])
incomes = np.clip(incomes, 20_000, 2_000_000)

mean_inc   = incomes.mean()
median_inc = np.median(incomes)
mode_approx = stats.mode(incomes.astype(int)//5000*5000, keepdims=True).mode[0]

print("💰 Income Distribution Analysis")
print(f"  Mean:    ${mean_inc:>10,.0f}")
print(f"  Median:  ${median_inc:>10,.0f}")
print(f"  Std Dev: ${incomes.std():>10,.0f}")
print(f"  IQR:     ${np.percentile(incomes,75) - np.percentile(incomes,25):>10,.0f}")
print(f"  Skewness:{stats.skew(incomes):>10.3f}")
print()
print(f"  👆 Mean is {mean_inc/median_inc:.1f}x the median — classic right-skew signal!")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(incomes/1000, bins=60, color='#3498db', edgecolor='white', lw=0.5)
axes[0].axvline(mean_inc/1000, color='red', lw=2.5, linestyle='--', label=f'Mean ${mean_inc/1000:.0f}k')
axes[0].axvline(median_inc/1000, color='green', lw=2.5, linestyle='-', label=f'Median ${median_inc/1000:.0f}k')
axes[0].set_title('💰 Income Distribution', fontweight='bold')
axes[0].set_xlabel('Income ($thousands)'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].set_xlim(0, 500)

axes[1].boxplot(incomes/1000, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6))
axes[1].set_title('Box Plot (IQR View)', fontweight='bold')
axes[1].set_ylabel('Income ($thousands)'); axes[1].set_ylim(0, 500)

# Compare: symmetric vs skewed
sym = np.random.normal(50000, 15000, 1000)
skew_data = np.random.lognormal(10.8, 0.5, 1000)
for data, color, label in [(sym, '#3498db', 'Symmetric'), (skew_data/1000, '#e74c3c', 'Right-Skewed')]:
    axes[2].hist(data/1000 if label == 'Symmetric' else data,
                 bins=40, alpha=0.5, density=True, color=color, label=label)
axes[2].set_title('Symmetric vs Skewed: Mean≠Median', fontweight='bold')
axes[2].set_xlabel('Value (thousands)')
axes[2].legend()

plt.tight_layout()
plt.savefig('ch12_descriptive.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Five-Number Summary and Outlier Detection
np.random.seed(42)
data = pd.Series(np.concatenate([np.random.normal(100, 15, 200), [200, 220, -10]]))

Q1  = data.quantile(0.25)
Q3  = data.quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outliers = data[(data < lower_fence) | (data > upper_fence)]

print("📊 Five-Number Summary")
print(f"  Min:    {data.min():.1f}")
print(f"  Q1:     {Q1:.1f}")
print(f"  Median: {data.median():.1f}")
print(f"  Q3:     {Q3:.1f}")
print(f"  Max:    {data.max():.1f}")
print(f"  IQR:    {IQR:.1f}")
print(f"  Fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
print(f"  Outliers: {sorted(outliers.values)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data.hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
for o in outliers:
    axes[0].axvline(o, color='red', lw=2, linestyle='--')
axes[0].set_title('Histogram with Outliers', fontweight='bold')

axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6),
                flierprops=dict(marker='o', color='red', markersize=8))
axes[1].set_title('Box Plot: Outliers as Red Dots', fontweight='bold')
plt.tight_layout()
plt.savefig('ch12_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comprehensive summary statistics: pandas describe() + extras
np.random.seed(42)
datasets = {
    'Normal(50,10)':     np.random.normal(50, 10, 500),
    'Right-Skewed':      np.random.lognormal(3.8, 0.5, 500),
    'Left-Skewed':       100 - np.random.lognormal(3.5, 0.4, 500),
    'Bimodal':           np.concatenate([np.random.normal(30, 5, 250), np.random.normal(70, 5, 250)]),
}

print(f"{'Metric':<20} {'Normal':>14} {'Right-Skew':>14} {'Left-Skew':>14} {'Bimodal':>12}")
print("-" * 76)
metrics = {
    'Mean':     lambda d: d.mean(),
    'Median':   lambda d: np.median(d),
    'Std Dev':  lambda d: d.std(),
    'IQR':      lambda d: np.percentile(d,75)-np.percentile(d,25),
    'Skewness': lambda d: stats.skew(d),
    'Kurtosis': lambda d: stats.kurtosis(d),
}
names = list(datasets.keys())
for metric_name, func in metrics.items():
    vals = [func(d) for d in datasets.values()]
    row = " ".join(f"{v:>14.2f}" for v in vals)
    print(f"{metric_name:<20} {row}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, data) in zip(axes, datasets.items()):
    ax.hist(data, bins=30, color='#3498db', edgecolor='white', density=True)
    ax.axvline(data.mean(), color='red', lw=2, linestyle='--', label='Mean')
    ax.axvline(np.median(data), color='green', lw=2, linestyle='-', label='Median')
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle("Descriptive Statistics Across Different Distributions", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ch12_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**P1:** Data: [12, 15, 14, 10, 98, 13, 16, 11]. Find mean, median, IQR. Is 98 an outlier?
**P2:** A distribution has mean=70, median=60. What does this tell you about its shape?
**P3:** Which is more informative about "typical salary" — mean or median? Why?

<details><summary>💡 Solutions</summary>

**P1:** Mean=(189/8)=23.6, Median=(13+14)/2=13.5, IQR=15-11=4.
Outlier check: Q3+1.5×IQR=15+6=21. Since 98>21, **yes, 98 is an outlier**.

**P2:** Mean>Median → **right-skewed** (long right tail pulling mean up).

**P3:** **Median** — it's not affected by extreme values (like a CEO's $10M salary).
</details>

In [ ]:
data_p1 = np.array([12, 15, 14, 10, 98, 13, 16, 11])
Q1, Q3 = np.percentile(data_p1, [25, 75])
IQR = Q3 - Q1
print(f"Mean={data_p1.mean():.1f}, Median={np.median(data_p1)}, IQR={IQR}")
print(f"Upper fence = {Q3 + 1.5*IQR:.1f}")
print(f"Is 98 an outlier? {98 > Q3 + 1.5*IQR}")

## 🎯 Episode Recap & Tier 1 Summary

**This episode's takeaways:**
1. Use **median** for skewed data or when outliers are present.
2. **IQR** is a robust measure of spread; use it to detect outliers.
3. Skewness and kurtosis describe the **shape** beyond mean and variance.

## 🏆 Tier 1 Complete!

You've covered the foundations of probability and statistics:
- ✅ What probability means and its three types
- ✅ Rules: complement, addition, multiplication
- ✅ Conditional probability and Bayes' Theorem
- ✅ Random variables: PMF, PDF, CDF
- ✅ Expected value and variance
- ✅ Five key distributions: Bernoulli, Binomial, Geometric, Poisson, Normal, Exponential
- ✅ Central Limit Theorem — the keystone of statistical inference
- ✅ Descriptive statistics: summarizing data meaningfully

**Choose your path:**
- 🎓 [Track 1 — Students: Combinatorics, Hypothesis Testing, Exam Prep]
- 💻 [Track 2 — Developers: Monte Carlo, A/B Testing, Probabilistic Systems]
- 📈 [Track 3 — Data Scientists: MLE, Bias-Variance, Causal Inference]
- ⚙️ [Track 4 — Engineers: Reliability, Queuing Theory, Six Sigma]